# 04 — SPURS stability scores

SPURS reports raw ΔΔ*G* (wild-type substitutions ≈ 0). Notebook 05 uses
**−SPURS** so that stabilizing substitutions rank higher.

Inference (`get_SPURS_from_hub` + `parse_pdb_resolved_only`) runs **only on
CUDA**. Without a GPU this notebook loads the cached CSV under `scores/`.

Do not `pip install spurs` from PyPI. See the README for the
`fix/py311-inference-beta` install.

## Resolve scores (GPU scan or cached table)

In [1]:
from IPython.display import display

from dualde import load_example
from dualde.mapping import load_structural_mapping_json, read_fasta, select_mapping_entry
from dualde.scores import add_neg_spurs, cuda_is_available, load_spurs_table, run_spurs_scan

EXAMPLE, cfg = load_example()
OUT = EXAMPLE / "outputs"
OUT.mkdir(parents=True, exist_ok=True)
PDB_ID = cfg["pdb_id"]
CHAIN = cfg["chain"]
CACHE = EXAMPLE / "scores" / f"{PDB_ID}_spurs.csv"

if cuda_is_available():
    print("CUDA GPU detected; running SPURS.")
    _, uniprot_seq = read_fasta(EXAMPLE / f"{cfg['protein']}.fasta")
    entry = select_mapping_entry(
        load_structural_mapping_json(EXAMPLE / "mapping" / f"{PDB_ID}_{CHAIN}.json"),
        pdb_id=PDB_ID,
        chain=CHAIN,
        uniprot_id=cfg["uniprot_id"],
    )
    try:
        spurs = run_spurs_scan(
            str(EXAMPLE / "pdb" / f"{PDB_ID}_cleaned.pdb"),
            entry,
            chain=CHAIN,
            uniprot_sequence=uniprot_seq,
        )
    except ImportError as err:
        print(err)
        print("Falling back to the precomputed table.")
        spurs = load_spurs_table(CACHE)
else:
    print("No CUDA GPU; loading precomputed SPURS ΔΔG.")
    spurs = load_spurs_table(CACHE)

spurs = add_neg_spurs(spurs)
spurs.to_csv(OUT / f"{PDB_ID}_spurs.csv", index=False)
print("wrote", OUT / f"{PDB_ID}_spurs.csv")

No CUDA GPU; loading precomputed SPURS ΔΔG.
wrote /lustre/scratch125/gengen/projects_v2/sgb_enzyme_design/evedesign_prelim/dualde/examples/NUDT15_HUMAN/outputs/5lpg_spurs.csv


## Inspect

In [2]:
print("shape:", spurs.shape)
print("columns:", list(spurs.columns))
print(
    "UniProt positions:",
    int(spurs["uniprot_position"].min()),
    "–",
    int(spurs["uniprot_position"].max()),
)
print("n unique positions:", spurs["uniprot_position"].nunique())
display(spurs.head())

shape: (3120, 7)
columns: ['mutant', 'spurs', 'wt', 'subs', 'uniprot_position', 'pdb_position', 'neg_spurs']
UniProt positions: 9 – 164
n unique positions: 156


,mutant,spurs,wt,subs,uniprot_position,pdb_position,neg_spurs
0,G9A,0.193578,G,A,9,9,-0.193578
1,G9C,-0.219615,G,C,9,9,0.219615
2,G9D,0.290070,G,D,9,9,-0.290070
3,G9E,0.188686,G,E,9,9,-0.188686
4,G9F,0.197176,G,F,9,9,-0.197176


In [3]:
wt = spurs.loc[spurs["wt"] == spurs["subs"], "spurs"]
print(
    "wild-type diagonal (expect ~0): n =",
    len(wt),
    "mean =",
    float(wt.mean()),
    "max |ΔΔG| =",
    float(wt.abs().max()),
)
display(spurs[["spurs", "neg_spurs"]].describe().T)

wild-type diagonal (expect ~0): n = 156 mean = 0.0 max |ΔΔG| = 0.0


,count,mean,std,min,25%,50%,75%,max
spurs,3120.0,0.964882,1.00011,-1.413262,0.194673,0.713557,1.521858,4.373418
neg_spurs,3120.0,-0.964882,1.00011,-4.373418,-1.521858,-0.713557,-0.194673,1.413262


In [4]:
site_pos = int(spurs["uniprot_position"].median())
display(
    spurs.loc[
        spurs["uniprot_position"] == site_pos,
        ["mutant", "wt", "subs", "spurs", "neg_spurs"],
    ].head(10)
)

,mutant,wt,subs,spurs,neg_spurs
1540,E86A,E,A,0.192968,-0.192968
1541,E86C,E,C,0.174564,-0.174564
1542,E86D,E,D,0.304050,-0.304050
1543,E86E,E,E,0.000000,-0.000000
1544,E86F,E,F,0.297901,-0.297901
1545,E86G,E,G,0.520681,-0.520681
1546,E86H,E,H,0.304258,-0.304258
1547,E86I,E,I,0.457559,-0.457559
1548,E86K,E,K,0.365935,-0.365935
1549,E86L,E,L,0.337769,-0.337769
